# JSON to Excel Export (Template-First)

This notebook exports one workbook per mold family from `mold_data.json`
(schemaVersion 5.0 — components keyed by governed Mold ID per the
standard in `docs/reference/`).

Rules implemented:
- Always load the latest template workbook.
- Follow template sheet/table/range names as source of truth.
- Component sheets are named by short Mold ID (e.g. `MS-PRI-B`), matching
  the Summary "Mold ID" column so the qty INDIRECT formulas resolve.
- Keep null `vendorShortName` as blank.
- Ignore non-numeric mold-size keys in sizing maps.

In [ ]:
# ----------------------------
# Configuration — edit here before running
# ----------------------------
JSON_PATH      = "../data/mold_data.json"
TEMPLATE_PATH  = "../docs/templates/MoldFamily_(Template).xlsx"
OUTPUT_DIR     = "../data/output"
MAX_FAMILIES   = 20   # e.g. 3 for a quick smoke-test; None = all
SKIP_EXISTING  = False  # True = skip families whose .xlsx already exists in OUTPUT_DIR

In [ ]:
# Clear stale win32com gen_py cache — fixes AttributeError: CLSIDToPackageMap
import shutil
from pathlib import Path
import win32com.client.gencache as _gencache

_gen_py = Path(_gencache.GetGeneratePath())
if _gen_py.exists():
    shutil.rmtree(_gen_py, ignore_errors=True)
    _gen_py.mkdir(parents=True, exist_ok=True)
    print(f"Cleared gen_py cache: {_gen_py}")
else:
    print("gen_py cache already clean")

In [ ]:
import json
import re
import shutil
import time
from pathlib import Path

import win32com.client as win32

from mold_v3_mapping import short_mold_id


def to_stripped(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def as_number_or_none(value):
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return None


def is_numeric_size_key(value):
    if value is None:
        return False
    try:
        float(str(value).strip())
        return True
    except Exception:
        return False


INVALID_SHEET_CHARS = r"[\[\]\:\*\?\/\\]"


def safe_sheet_name(name):
    name = "Sheet" if name is None else str(name)
    name = re.sub(INVALID_SHEET_CHARS, "-", name).strip()
    if len(name) > 31:
        name = name[:31]
    return name or "Sheet"


def comp_short_id(comp):
    """Short Mold ID (e.g. 'MS-PRI-B') rebuilt from component attributes —
    used for Excel sheet names and the Summary Mold ID column."""
    return short_mold_id(comp.get("type"), comp.get("position"), comp.get("stage"))


def mold_size_to_row(size_value):
    size_f = float(size_value)
    step = (size_f - 1.0) / 0.5
    rounded = round(step)
    if abs(step - rounded) > 1e-9:
        return None
    row = 9 + int(rounded)   # row 9 = mold size 1.0 (shifted +1 for new Mold Family row)
    if row < 9 or row > 43:
        return None
    return row

In [ ]:
def _normalize_table_values(values):
    if values is None:
        return []
    if not isinstance(values, tuple):
        return [[values]]
    if len(values) == 0:
        return []
    first = values[0]
    if isinstance(first, tuple):
        return [list(r) for r in values]
    return [list(values)]


def read_listobject_records(ws, table_name):
    lo = ws.ListObjects(table_name)
    rows = _normalize_table_values(lo.Range.Value)
    if not rows:
        return []
    headers = rows[0]
    out = []
    for row in rows[1:]:
        out.append({headers[i]: row[i] if i < len(row) else None for i in range(len(headers))})
    return out


def build_lookup_maps(ws_master):
    vendor_rows  = read_listobject_records(ws_master, "_dimVendor")
    factory_rows = read_listobject_records(ws_master, "_dimFactory")

    vendor_short_by_id = {}
    for row in vendor_rows:
        vendor_id    = to_stripped(row.get("Vendor ID"))
        vendor_short = to_stripped(row.get("Vendor Short Name"))
        if vendor_id and vendor_short:
            vendor_short_by_id[vendor_id] = vendor_short

    factory_name_by_number = {}
    for row in factory_rows:
        number = row.get("Factory Number")
        name   = to_stripped(row.get("Factory Name"))
        if number is None or not name:
            continue
        try:
            key = str(int(float(number)))
        except Exception:
            key = str(number).strip()
        factory_name_by_number[key] = name

    return vendor_short_by_id, factory_name_by_number


def iter_component_records(fam):
    """Flat v5 structure: components keyed by full Mold ID, assets per component."""
    components = fam.get("components") or {}
    for mold_id, comp in components.items():
        for asset in comp.get("assets") or []:
            yield mold_id, comp, asset


def collect_summary_rows(fam, vendor_short_by_id, vendor_location_by_id=None):
    vendor_location_by_id = vendor_location_by_id or {}
    rows = []
    for mold_id, comp, asset in iter_component_records(fam):
        cap = asset.get("capacity") or {}

        vendor_id    = to_stripped(asset.get("vendorId"))
        vendor_short = (vendor_short_by_id.get(vendor_id) if vendor_id else None) \
                       or to_stripped(asset.get("vendorName")) or ""
        location = vendor_location_by_id.get(vendor_id) if vendor_id else None

        rows.append({
            # Summary col A carries the short Mold ID — must equal the
            # component sheet name for the F-column INDIRECT chain to resolve
            "component_code":  comp_short_id(comp),
            "sole_type":       comp.get("soleType"),
            "component_name":  to_stripped(comp.get("componentDescription")),
            "vendor_short_name": vendor_short,
            "location":        location,
            "total_mold_qty":  as_number_or_none(asset.get("totalMoldQty")),
            "ownership":       to_stripped(asset.get("ownership")),
            "condition":       to_stripped(asset.get("condition")),
            "mold_shop":       to_stripped(asset.get("moldShopCode")),
            "init_season":     to_stripped(asset.get("initSeason")),
            "daily_output":    as_number_or_none(cap.get("dailyOutput")),
            "mold_init_cost":  as_number_or_none(asset.get("moldCost")),
            "comments":        to_stripped(asset.get("comments")),
        })
    return rows

In [ ]:
def write_summary(ws_summary, family_code, fam, vendor_short_by_id, warnings,
                  vendor_location_by_id=None):
    input_start_row = 7
    input_end_row   = 31

    ws_summary.Cells(1, 2).Value = family_code

    styles_by_brand = fam.get("stylesUsingThisFamily") or {}
    unique_brands   = list(styles_by_brand.keys())
    ws_summary.Cells(2, 2).Value = ", ".join(unique_brands) if unique_brands else None

    # Clear only key-in columns: A (Mold ID), D (Vendor Name), H-N (asset/capacity)
    # B/C/E/F/G are formula or lookup columns — do not touch them
    ws_summary.Range(f"A{input_start_row}:A{input_end_row}").ClearContents()
    ws_summary.Range(f"D{input_start_row}:D{input_end_row}").ClearContents()
    ws_summary.Range(f"H{input_start_row}:N{input_end_row}").ClearContents()
    ws_summary.Range("R7:T31").ClearContents()

    rows = collect_summary_rows(fam, vendor_short_by_id, vendor_location_by_id)
    capacity = input_end_row - input_start_row + 1
    if len(rows) > capacity:
        warnings.append(f"{family_code}: Summary rows truncated from {len(rows)} to {capacity}.")
        rows = rows[:capacity]

    if rows:
        n       = len(rows)
        end_row = input_start_row + n - 1

        ws_summary.Range(f"A{input_start_row}:A{end_row}").Value = \
            tuple((r["component_code"],) for r in rows)
        ws_summary.Range(f"D{input_start_row}:D{end_row}").Value = \
            tuple((r["vendor_short_name"] or None,) for r in rows)
        ws_summary.Range(f"H{input_start_row}:N{end_row}").Value = tuple(
            (r["ownership"], r["condition"], r["mold_shop"],
             r["init_season"], r["daily_output"], r["mold_init_cost"], r["comments"])
            for r in rows
        )

    # Write styles section (R7:T31 — style name, outsole flag, midsole flag)
    style_rows = []
    for brand, styles_list in styles_by_brand.items():
        for s in (styles_list or []):
            style_name = to_stripped(s.get("styleName"))
            if not style_name:
                continue
            sole_types = s.get("soleTypes") or []
            style_rows.append((
                style_name,
                "Outsole" in sole_types,
                "Midsole" in sole_types,
            ))

    style_capacity = 25  # rows 7–31
    if len(style_rows) > style_capacity:
        warnings.append(f"{family_code}: Styles truncated from {len(style_rows)} to {style_capacity}.")
        style_rows = style_rows[:style_capacity]

    if style_rows:
        ns = len(style_rows)
        ws_summary.Range(f"R7:T{7 + ns - 1}").Value = tuple(style_rows)


def write_component_sheet(ws_inv, family_code, comp,
                          vendor_short_by_id, factory_name_by_number, warnings):
    mold_id  = to_stripped(comp.get("moldId"))
    short_id = comp_short_id(comp)

    ws_inv.Cells(2, 2).Value = family_code
    ws_inv.Cells(3, 2).Value = mold_id
    ws_inv.Cells(4, 2).Value = to_stripped(comp.get("componentDescription")) or None
    ws_inv.Cells(5, 2).Value = to_stripped(comp.get("designCompound")) or None

    # Clear inventory area: A=shoe sizes, B-E=4 vendor qty columns
    ws_inv.Range("A9:E43").ClearContents()

    assets = comp.get("assets") or []
    vendor_order = []
    for a in assets:
        vid    = to_stripped(a.get("vendorId"))
        vshort = (vendor_short_by_id.get(vid) if vid else None) \
                 or to_stripped(a.get("vendorName")) or ""
        if vshort not in vendor_order:
            vendor_order.append(vshort)

    max_vendor_cols = 4  # B–E
    if len(vendor_order) > max_vendor_cols:
        warnings.append(
            f"{family_code}/{short_id}: Vendor columns truncated from {len(vendor_order)} to {max_vendor_cols}."
        )
        vendor_order = vendor_order[:max_vendor_cols]

    vendor_to_col = {}
    header_row    = [None] * max_vendor_cols
    for i, value in enumerate(vendor_order):
        vendor_to_col[value] = 2 + i  # B=2
        header_row[i]        = value if value else None

    # Write vendor headers to row 8, B8:E8
    ws_inv.Range("B8:E8").Value = (tuple(header_row),)

    # Key by full shoe-sizes pattern, not just base size.
    # Two entries sharing a base size but different coverage counts (e.g. ["3.5"] vs ["3.5","4","4.5"])
    # are distinct physical mold types and must occupy separate rows.
    # pattern_meta: shoe_sizes_text → {"base_f": float, "n": int, "qty_by_col": {col: qty}}
    pattern_meta = {}

    for asset in assets:
        vid          = to_stripped(asset.get("vendorId"))
        vendor_short = (vendor_short_by_id.get(vid) if vid else None) \
                       or to_stripped(asset.get("vendorName")) or ""
        if vendor_short not in vendor_to_col:
            continue
        target_col = vendor_to_col[vendor_short]

        for entry in asset.get("sizeCoverage") or []:
            shoe_sizes = entry.get("shoeSizes") or []
            if not shoe_sizes:
                continue
            qty = as_number_or_none(entry.get("qty"))
            if qty is None:
                continue

            pattern_key = ", ".join(shoe_sizes)
            if pattern_key not in pattern_meta:
                pattern_meta[pattern_key] = {
                    "base_f":     float(shoe_sizes[0]),
                    "n":          len(shoe_sizes),
                    "qty_by_col": {},
                }
            pattern_meta[pattern_key]["qty_by_col"][target_col] = \
                pattern_meta[pattern_key]["qty_by_col"].get(target_col, 0.0) + qty

    # Write sequentially from row 9: sort by (base_f, n) — same base size, 1:1 before 1:3
    sorted_patterns = sorted(pattern_meta, key=lambda k: (pattern_meta[k]["base_f"],
                                                           pattern_meta[k]["n"]))
    inv_capacity = 35  # rows 9–43
    if len(sorted_patterns) > inv_capacity:
        warnings.append(
            f"{family_code}/{short_id}: Inventory rows truncated from {len(sorted_patterns)} to {inv_capacity}."
        )
        sorted_patterns = sorted_patterns[:inv_capacity]

    for seq_row, pattern_key in enumerate(sorted_patterns, start=9):
        ws_inv.Cells(seq_row, 1).Value = pattern_key  # col A
        for col, qty in pattern_meta[pattern_key]["qty_by_col"].items():
            ws_inv.Cells(seq_row, col).Value = qty

    # Sourcing rules: H=factory name (key-in), K=vendor short name (key-in)
    ws_inv.Range("H9:H43").ClearContents()
    ws_inv.Range("K9:K43").ClearContents()

    sourcing_rules = comp.get("sourcingRules") or []
    src_capacity   = 35  # rows 9–43
    if len(sourcing_rules) > src_capacity:
        warnings.append(
            f"{family_code}/{short_id}: Sourcing rules truncated from {len(sourcing_rules)} to {src_capacity}."
        )
        sourcing_rules = sourcing_rules[:src_capacity]

    if sourcing_rules:
        factory_vals = []
        vendor_vals  = []
        for rule in sourcing_rules:
            factory_number = rule.get("factoryNumber")
            if factory_number is None:
                factory_name = None
            else:
                try:
                    factory_key = str(int(float(factory_number)))
                except Exception:
                    factory_key = str(factory_number).strip()
                factory_name = factory_name_by_number.get(factory_key)

            vid          = to_stripped(rule.get("vendorId"))
            vendor_short = vendor_short_by_id.get(vid) if vid else None

            factory_vals.append((factory_name,))
            vendor_vals.append((vendor_short,))

        ns = len(sourcing_rules)
        ws_inv.Range(f"H9:H{9 + ns - 1}").Value = tuple(factory_vals)
        ws_inv.Range(f"K9:K{9 + ns - 1}").Value = tuple(vendor_vals)

In [ ]:
def export_one_family(app, template_abs, family_code, fam, out_path,
                      vendor_short_by_id, factory_name_by_number, vendor_location_by_id=None):
    warnings = []

    wb = app.Workbooks.Open(template_abs)
    try:
        ws_summary      = wb.Worksheets("Summary")
        ws_inv_template = wb.Worksheets("{Component}")

        write_summary(ws_summary, family_code, fam, vendor_short_by_id, warnings,
                      vendor_location_by_id)

        components  = fam.get("components") or {}
        sheet_names = {s.Name for s in wb.Worksheets}
        created     = set()

        for mold_id, comp in components.items():
            # Copy with a single positional Before argument. Named args
            # (Copy(After=...)) are silently dropped by win32com dynamic
            # dispatch — Excel then copies the sheet into a brand-new
            # workbook — and Copy(None, after) passes Before=Null, which
            # Excel treats as "insert before the first sheet". The new
            # sheet is then found by name-set difference, which is
            # independent of where Excel inserted it.
            before_names = {s.Name for s in wb.Worksheets}
            ws_inv_template.Copy(wb.Worksheets(wb.Worksheets.Count))
            new_names = [s.Name for s in wb.Worksheets if s.Name not in before_names]
            if len(new_names) != 1:
                raise RuntimeError(f"{family_code}: worksheet copy failed ({new_names})")
            new_ws = wb.Worksheets(new_names[0])

            # Sheet name = short Mold ID (e.g. "MS-PRI-B") — must match
            # the Summary col A value for the INDIRECT chain to resolve
            target = safe_sheet_name(comp_short_id(comp))
            base   = target
            suffix = 1
            while target in sheet_names or target in created:
                suffix += 1
                target = safe_sheet_name(f"{base}_{suffix}")
            new_ws.Name = target
            created.add(target)
            sheet_names.add(target)

            write_component_sheet(new_ws, family_code, comp,
                                  vendor_short_by_id, factory_name_by_number, warnings)

        app.DisplayAlerts = False
        ws_inv_template.Delete()
        app.DisplayAlerts = True

        wb.SaveAs(str(out_path), FileFormat=51)
    finally:
        wb.Close(SaveChanges=False)

    return warnings


def export_all_families(json_path, template_path, output_dir,
                        max_families=None, skip_existing=False):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    if data.get("schemaVersion") != "5.0":
        raise ValueError(
            f"Expected schemaVersion 5.0, got {data.get('schemaVersion')!r} — "
            "re-run data_processing.ipynb first."
        )
    families = data.get("families") or {}
    if not families:
        raise ValueError("No families found in JSON.")

    # Build vendor location lookup from JSON reference section
    vendor_ref_data       = data.get("reference", {}).get("vendors", {})
    vendor_location_by_id = {
        vid: ((vdata.get("location") or {}).get("name") or
              (vdata.get("location") or {}).get("code") or "")
        for vid, vdata in vendor_ref_data.items()
    }

    out_dir      = Path(output_dir).resolve()
    template_abs = str(Path(template_path).resolve())
    out_dir.mkdir(parents=True, exist_ok=True)

    app = win32.DispatchEx("Excel.Application")
    app.Visible       = False
    app.DisplayAlerts = False

    written_files = []
    skipped_files = []
    all_warnings  = []
    total = sum(1 for i, _ in enumerate(families, 1)
                if max_families is None or i <= max_families)

    try:
        template_wb = app.Workbooks.Open(template_abs)
        vendor_short_by_id, factory_name_by_number = build_lookup_maps(
            template_wb.Worksheets("_Master_Ref")
        )
        template_wb.Close(SaveChanges=False)

        for i, (family_code, fam) in enumerate(families.items(), start=1):
            if max_families is not None and i > max_families:
                break

            out_path = out_dir / f"{family_code}.xlsx"

            if skip_existing and out_path.exists():
                skipped_files.append(str(out_path))
                print(f"[{i}/{total}] SKIP {family_code}   ", end="\r", flush=True)
                continue

            print(f"[{i}/{total}] {family_code}   ", end="\r", flush=True)

            warnings = export_one_family(
                app=app,
                template_abs=template_abs,
                family_code=family_code,
                fam=fam,
                out_path=out_path,
                vendor_short_by_id=vendor_short_by_id,
                factory_name_by_number=factory_name_by_number,
                vendor_location_by_id=vendor_location_by_id,
            )
            written_files.append(str(out_path))
            all_warnings.extend(warnings)

        print()

    finally:
        app.Quit()

    return written_files, skipped_files, all_warnings

In [7]:
t0 = time.time()

files, skipped, warnings = export_all_families(
    json_path=JSON_PATH,
    template_path=TEMPLATE_PATH,
    output_dir=OUTPUT_DIR,
    max_families=MAX_FAMILIES,
    skip_existing=SKIP_EXISTING,
)

elapsed = time.time() - t0
print(f"Exported {len(files)} workbook(s) in {elapsed:.1f}s  |  Skipped {len(skipped)}  |  Warnings {len(warnings)}")
print(f"Output: {OUTPUT_DIR}")
for path in files[:10]:
    print(" -", path)
if len(files) > 10:
    print(f" ... and {len(files) - 10} more")
for w in warnings[:30]:
    print(" !", w)
if len(warnings) > 30:
    print(f" ... and {len(warnings) - 30} more")

[20/20] SA-2652     
Exported 20 workbook(s) in 25.0s  |  Skipped 0  |  Warnings 0
Output: ../data/output
 - D:\Project\osms-mold-master-data\data\output\SA-1318.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2017.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2017-E.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2253.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2255.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2255-4E.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2301.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2313.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2408.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2451.xlsx
 ... and 10 more
